In [ ]:
import nomad.filters as filters 
import nomad.stop_detection.grid_based as GRID_BASED

fig, (ax_map, ax_barcode) = plt.subplots(2, 1, figsize=(6,6.5),
                                         gridspec_kw={'height_ratios':[10,1]})

colors = {
    'home': 'lightgrey',
    'work': 'lightgrey', 
    'retail': 'lightgrey', 
    'park': 'lightgrey',
    'street': 'white',
    'default': 'lightgrey'
}


# normal
# labels = LACHESIS.lachesis_labels(data=Charlie.sparse_traj, dt_max=30, delta_roam=3, dur_min=5)

# many visits one point
# labels = LACHESIS.lachesis_labels(data=Charlie.sparse_traj, dt_max=30, delta_roam=2, dur_min=5)

# temporal overlap DBSCAN
labels = DBSCAN.ta_dbscan_labels(Charlie.sparse_traj, time_thresh=180, dist_thresh=1, min_pts=3)

# splitting
# labels = LACHESIS.lachesis_labels(data=Charlie.sparse_traj, dt_max=30, delta_roam=1, dur_min=5)

# missingness
# labels = LACHESIS.lachesis_labels(data=Charlie.sparse_traj, dt_max=4, delta_roam=1, dur_min=5)


city.plot_city(ax_map, doors=True, address=False, zorder=0, colors=colors)

# ax_map.plot(Charlie.trajectory['x'], Charlie.trajectory['y'], 
#            color='red', linewidth=0.8, alpha=0.4, zorder=1)

x0, x1 = Charlie.trajectory.x.min(), Charlie.trajectory.x.max()
y0, y1 = Charlie.trajectory.y.min(), Charlie.trajectory.y.max()
pad_x = (x1 - x0) * 0.5 / 2
pad_y = (y1 - y0) * 0.5 / 2
ax_map.set_xlim(13.4, 21.5)
ax_map.set_ylim(y0 - pad_y, y1 + pad_y)
ax_map.set_xticks([])
ax_map.set_yticks([])
# ax_map.set_title('A) Correct', fontsize=14, pad=10, fontweight='bold')

# Manually plot colored points by cluster to match barcode
cmap_obj = plt.get_cmap('terrain')
unique_clusters = labels.unique()
unique_clusters = unique_clusters[unique_clusters >= 0]
num_clusters = len(unique_clusters)

# Plot noise points (black)
noise_mask = labels == -1
if noise_mask.any():
    noise_data = Charlie.sparse_traj[noise_mask]
    ax_map.scatter(noise_data['x'], noise_data['y'], 
                  s=30, color='black', alpha=0.5, zorder=2, edgecolor='white', linewidth=0.5)

# Plot each cluster with matching colors
if num_clusters > 0:
    for cid in unique_clusters:
        cluster_mask = labels == cid
        if cluster_mask.any():
            cluster_data = Charlie.sparse_traj[cluster_mask]
            col = cmap_obj(cid / num_clusters)
            ax_map.scatter(cluster_data['x'], cluster_data['y'], 
                          s=80, color=col, alpha=1, zorder=3, linewidth=0.5)

# Add small black dots on top for all points
ax_map.scatter(Charlie.sparse_traj['x'], Charlie.sparse_traj['y'], 
              s=6, color='black', alpha=1, zorder=4)

# ax_map.set_title('Lachesis Stop Detection', fontsize=14, pad=10, fontweight='bold')

# Barcode with same colormap
plot_time_barcode_colored(Charlie.sparse_traj['timestamp'], labels, ax=ax_barcode, 
                          cmap='terrain', set_xlim=True)

plt.tight_layout()
plt.savefig('missing_clusters.svg', format='svg')
plt.show()